In [ ]:
# -----------------------------
# Dataset generator (optional)
# -----------------------------
# This cell generates a synthetic rank_eval.csv consistent with ranking constraints.

import json
import random
from datetime import datetime, timedelta, timezone
import pandas as pd

random.seed(42)

STOPS = [
    "Gulberg", "DHA", "Johar Town", "Model Town", "Wapda Town", "Bahria Town",
    "Saddar", "Clifton", "G-10", "F-11", "I-8", "Blue Area"
]

VEHICLES = [
    {"vehicle_company": "Honda", "vehicle_type": "Car", "vehicle_color": "White", "vehicle_seats": 4, "vehicle_fuel_type": "Petrol"},
    {"vehicle_company": "Toyota", "vehicle_type": "Car", "vehicle_color": "Black", "vehicle_seats": 4, "vehicle_fuel_type": "Petrol"},
    {"vehicle_company": "Suzuki", "vehicle_type": "Car", "vehicle_color": "Silver", "vehicle_seats": 4, "vehicle_fuel_type": "Petrol"},
    {"vehicle_company": "Yamaha", "vehicle_type": "Bike", "vehicle_color": "Red", "vehicle_seats": 2, "vehicle_fuel_type": "Petrol"},
]


def pick_route():
    a, b = random.sample(STOPS, 2)
    return a, b


def iso(dt):
    return dt.replace(microsecond=0).isoformat()


def _median_int(xs: list[int]) -> int:
    if not xs:
        return 0
    ys = sorted(int(x) for x in xs)
    return int(ys[len(ys) // 2])


def _clamp_int(x: int, lo: int, hi: int) -> int:
    return max(lo, min(hi, int(x)))


def _clamp_float(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, float(x)))


def _route_distance_km(fr: str, to: str) -> float:
    base = 5.0 + (abs(hash(_clamp_int(hash(fr), -10**9, 10**9)) - hash(_clamp_int(hash(to), -10**9, 10**9))) % 450) / 10.0
    return _clamp_float(base, 3.0, 500.0)


def _duration_from_distance(distance_km: float) -> int:
    speed_kmh = 55.0
    minutes = int(round((distance_km / max(1e-6, speed_kmh)) * 60.0))
    return max(5, minutes)


def make_history(user_id, now, n=10):
    fav_from, fav_to = pick_route()
    fav_vehicle = random.choice(VEHICLES)

    bookings = []
    for i in range(n):
        dt = now - timedelta(days=random.randint(1, 30), hours=random.randint(0, 23))
        if random.random() < 0.7:
            fr, to = fav_from, fav_to
            veh = fav_vehicle
        else:
            fr, to = pick_route()
            veh = random.choice(VEHICLES)

        bookings.append({
            "trip_id": f"H{user_id}_{i}",
            "from_stop_name": fr,
            "to_stop_name": to,
            "total_fare": random.randint(200, 800),
            "number_of_seats": random.choice([1, 1, 1, 2]),
            "finalized_at": iso(dt),
            **veh,
        })
    return bookings, (fav_from, fav_to), fav_vehicle


def stop_breakdown_for(fr, to, price, distance_km: float, duration_minutes: int):
    return [{"from_stop_name": fr, "to_stop_name": to, "price": int(price), "distance_km": float(distance_km), "duration_minutes": int(duration_minutes)}]


def _pick_gender_preference_for_user(user_gender: str | None, *, positive: bool) -> str:
    ug = (user_gender or "").strip().title()
    if ug not in ("Male", "Female"):
        ug = ""

    if positive:
        if ug:
            return random.choices(["Any", ug], weights=[0.85, 0.15])[0]
        return "Any"

    return random.choices(["Any", "Male", "Female"], weights=[0.80, 0.10, 0.10])[0]


def make_candidate(
    trip_id,
    now,
    fr,
    to,
    user_gender: str | None,
    good=False,
    fav_vehicle=None,
    fixed_is_negotiable: bool | None = None,
    positive: bool = False,
    same_route_negative: bool = False,
    user_price_median: int | None = None,
    route_distance_km: float | None = None,
    positive_departure_dt: datetime | None = None,
):
    dist_km = float(route_distance_km) if route_distance_km is not None else float(_route_distance_km(fr, to))
    duration_min = _duration_from_distance(dist_km)

    if positive:
        dep = now + timedelta(hours=random.randint(1, 4))
    else:
        dep = now + timedelta(hours=random.randint(1, 36))

    if positive_departure_dt is not None and same_route_negative:
        if dep < positive_departure_dt:
            dep = positive_departure_dt + timedelta(minutes=random.randint(5, 240))

    base_price = random.randint(250, 750)

    if user_price_median is not None and user_price_median > 0:
        if positive:
            price = int(round(random.gauss(mu=user_price_median, sigma=max(30.0, 0.12 * user_price_median))))
        else:
            price = int(round(random.gauss(mu=user_price_median, sigma=max(80.0, 0.25 * user_price_median))))
    else:
        price = base_price

    if good:
        driver_rating = round(random.uniform(4.2, 5.0), 1)
        veh = fav_vehicle if fav_vehicle else random.choice(VEHICLES)
    else:
        driver_rating = round(random.uniform(2.5, 4.8), 1)
        veh = random.choice(VEHICLES)

        if same_route_negative:
            penalty = random.choice(["rating", "late", "price"])
            if penalty == "rating":
                driver_rating = min(driver_rating, 4.0)
            elif penalty == "late":
                dep = max(dep, now + timedelta(hours=random.randint(8, 48)))
                if positive_departure_dt is not None:
                    dep = max(dep, positive_departure_dt + timedelta(minutes=random.randint(5, 360)))
            else:
                price = price + random.randint(120, 260)

    gender_pref = _pick_gender_preference_for_user(user_gender, positive=positive)
    is_negotiable = fixed_is_negotiable if fixed_is_negotiable is not None else bool(random.getrandbits(1))

    price_per_seat = max(100, int(price))

    cand = {
        "trip_id": trip_id,
        "origin": fr,
        "destination": to,
        "departure_time": iso(dep),
        "price_per_seat": price_per_seat,
        "available_seats": random.randint(1, 4),
        "gender_preference": gender_pref,
        "is_negotiable": is_negotiable,
        "driver_rating": float(driver_rating),
        "total_distance_km": float(dist_km),
        "total_duration_minutes": int(duration_min),
        "stop_breakdown": stop_breakdown_for(fr, to, max(100, int(price_per_seat / 3)), dist_km, duration_min),
        **veh,
    }
    return cand


def generate_csv(
    out_path: str,
    num_events=5000,
    candidates_per_event=50,
    history_len=15,
):
    rows = []
    for e in range(num_events):
        event_id = f"E{e}"
        user_id = random.randint(1, 400)
        user_gender = random.choice(["Male", "Female"])
        passenger_rating = round(random.uniform(3.0, 5.0), 1)

        now = datetime.now(timezone.utc) - timedelta(days=random.randint(0, 60))
        history, (fav_from, fav_to), fav_vehicle = make_history(user_id, now, n=history_len)

        user_price_median = _median_int([int(h.get("total_fare") or 0) for h in history if int(h.get("total_fare") or 0) > 0])
        route_dist_km = _route_distance_km(fav_from, fav_to)

        pos_trip_id = f"T_POS_{event_id}"

        event_is_negotiable = bool(random.getrandbits(1))

        pos = make_candidate(
            pos_trip_id,
            now,
            fav_from,
            fav_to,
            user_gender=user_gender,
            good=True,
            fav_vehicle=fav_vehicle,
            fixed_is_negotiable=event_is_negotiable,
            positive=True,
            user_price_median=user_price_median,
            route_distance_km=route_dist_km,
        )

        pos_dep = datetime.fromisoformat(pos["departure_time"])

        candidates = [pos]
        for i in range(candidates_per_event - 1):
            if random.random() < 0.3:
                fr, to = fav_from, fav_to
                same_route_negative = True
                cand_dist = route_dist_km
            else:
                fr, to = pick_route()
                same_route_negative = False
                cand_dist = _route_distance_km(fr, to)

            cand = make_candidate(
                f"T_NEG_{event_id}_{i}",
                now,
                fr,
                to,
                user_gender=user_gender,
                good=False,
                fav_vehicle=fav_vehicle,
                fixed_is_negotiable=event_is_negotiable,
                positive=False,
                same_route_negative=same_route_negative,
                user_price_median=user_price_median,
                route_distance_km=cand_dist,
                positive_departure_dt=pos_dep,
            )
            candidates.append(cand)

        for c in candidates:
            label = 1 if c["trip_id"] == pos_trip_id else 0
            rows.append({
                "event_id": event_id,
                "user_id": user_id,
                "user_gender": user_gender,
                "passenger_rating": passenger_rating,
                "now_iso": iso(now),
                "history_bookings_json": json.dumps(history),
                "candidate_json": json.dumps(c),
                "label": label,
            })

    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    print("Saved:", out_path, "rows:", len(df), "events:", num_events)


In [ ]:
# -----------------------------
# Colab-ready evaluator (single notebook)
# -----------------------------
# This notebook can:
# 1) (Optional) generate rank_eval.csv into DRIVE_FOLDER
# 2) evaluate it using the embedded ranker core below

!pip -q install pandas numpy pydantic

from google.colab import drive
import os
import json
import math
from datetime import datetime, timezone
import pandas as pd

# -----------------------------
# Config
# -----------------------------
DRIVE_FOLDER = "/content/drive/MyDrive/Lets Go ml model evaluation code"
GENERATE_NEW_DATASET = False

NUM_EVENTS = 5000
CANDIDATES_PER_EVENT = 50
HISTORY_LEN = 15

MAX_EVENTS = 2000
MAX_CANDIDATES_PER_EVENT = None

# Debug printing
PRINT_WORST_N_EVENTS = 10
WORST_EVENT_MIN_BOOKED_RANK = 8
PRINT_TOP_K = 10

# -----------------------------
# Mount Drive
# -----------------------------
drive.mount("/content/drive")

os.makedirs(DRIVE_FOLDER, exist_ok=True)
CSV_PATH = os.path.join(DRIVE_FOLDER, "rank_eval.csv")
print("CSV will be saved/read at:\n ", CSV_PATH)

# Generate if requested OR if missing
if GENERATE_NEW_DATASET or (not os.path.exists(CSV_PATH)):
    if not GENERATE_NEW_DATASET:
        print("rank_eval.csv not found; auto-generating a new dataset...")
    generate_csv(
        out_path=CSV_PATH,
        num_events=NUM_EVENTS,
        candidates_per_event=CANDIDATES_PER_EVENT,
        history_len=HISTORY_LEN,
    )

assert os.path.exists(CSV_PATH), f"Failed to create rank_eval.csv at: {CSV_PATH}"

# -----------------------------
# Embedded ranker core (offline)
# -----------------------------
import math
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

from pydantic import BaseModel, Field


class UserPayload(BaseModel):
    user_id: int
    gender: Optional[str] = None
    passenger_rating: Optional[float] = 0.0


class HistoryBooking(BaseModel):
    trip_id: Optional[str] = None
    from_stop_name: Optional[str] = None
    to_stop_name: Optional[str] = None
    total_fare: Optional[int] = 0
    number_of_seats: Optional[int] = 0
    finalized_at: Optional[str] = None

    vehicle_company: Optional[str] = None
    vehicle_type: Optional[str] = None
    vehicle_color: Optional[str] = None
    vehicle_seats: Optional[int] = 0
    vehicle_fuel_type: Optional[str] = None


class StopBreakdownItem(BaseModel):
    from_stop_name: Optional[str] = None
    to_stop_name: Optional[str] = None
    price: Optional[int] = None


class CandidateTrip(BaseModel):
    trip_id: str
    origin: Optional[str] = None
    destination: Optional[str] = None
    departure_time: Optional[str] = None

    price_per_seat: Optional[int] = None
    available_seats: Optional[int] = None
    gender_preference: Optional[str] = None
    is_negotiable: Optional[bool] = None

    driver_rating: Optional[float] = 0.0

    vehicle_company: Optional[str] = None
    vehicle_type: Optional[str] = None
    vehicle_color: Optional[str] = None
    vehicle_seats: Optional[int] = 0
    vehicle_fuel_type: Optional[str] = None

    total_distance_km: Optional[float] = None
    total_duration_minutes: Optional[int] = None

    stop_breakdown: List[StopBreakdownItem] = Field(default_factory=list)


class RankedItem(BaseModel):
    trip_id: str
    score: float
    reasons: List[str] = Field(default_factory=list)


def _norm(s: Optional[str]) -> str:
    return " ".join((s or "").strip().lower().split())


def _tokens(s: str) -> set:
    return set([t for t in _norm(s).split(" ") if t])


def _jaccard(a: str, b: str) -> float:
    ta = _tokens(a)
    tb = _tokens(b)
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / max(1, len(ta | tb))


def _safe_float(x: Any, default: float = 0.0) -> float:
    try:
        return float(x)
    except Exception:
        return default


def _safe_int(x: Any, default: int = 0) -> int:
    try:
        return int(x)
    except Exception:
        return default


def _parse_iso_dt(s: Optional[str]) -> Optional[datetime]:
    if not s:
        return None
    try:
        return datetime.fromisoformat(str(s).replace("Z", "+00:00"))
    except Exception:
        return None


def _recency_weight(finalized_at_iso: Optional[str], now: datetime) -> float:
    dt = _parse_iso_dt(finalized_at_iso)
    if not dt:
        return 0.25
    age_h = max(0.0, (now - dt).total_seconds() / 3600.0)
    return math.exp(-age_h / 168.0)


def _gender_penalty(user_gender: Optional[str], trip_pref: Optional[str]) -> float:
    ug = (user_gender or "").strip()
    tp = (trip_pref or "Any").strip()
    if not ug:
        return 0.0
    if tp == "Any" or tp == "":
        return 0.0
    if (ug == "Male" and tp == "Female") or (ug == "Female" and tp == "Male"):
        return 0.90
    return 0.0


def _clamp(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, x))


def _trip_distance_km(c: CandidateTrip) -> float:
    d = _safe_float(c.total_distance_km, 0.0)
    if d > 0:
        return d
    return 20.0


def _distance_time_params(distance_km: float) -> Dict[str, float]:
    d = _clamp(distance_km, 1.0, 500.0)
    r = math.log1p(d) / math.log1p(500.0)

    sat_h = (1.0 * (1.0 - r)) + (12.0 * r)
    half_life_h = (6.0 * (1.0 - r)) + (72.0 * r)

    far_start_h = (18.0 * (1.0 - r)) + (120.0 * r)
    far_end_h = (72.0 * (1.0 - r)) + (168.0 * r)

    return {
        "sat_h": sat_h,
        "half_life_h": half_life_h,
        "far_start_h": far_start_h,
        "far_end_h": far_end_h,
    }


def _soonness_score(departure_iso: Optional[str], now: datetime, *, distance_km: float) -> float:
    dep = _parse_iso_dt(departure_iso)
    if not dep:
        return 0.0

    delta_h = (dep - now).total_seconds() / 3600.0
    if delta_h < 0:
        return 0.0

    p = _distance_time_params(distance_km)
    sat_h = p["sat_h"]
    half_life_h = max(1e-6, p["half_life_h"])

    if delta_h <= sat_h:
        return 1.0

    k = math.log(2.0) / half_life_h
    return math.exp(-k * (delta_h - sat_h))


def _build_user_pref(history: List[HistoryBooking], now: datetime) -> Dict[str, Any]:
    route_pairs: Dict[str, float] = {}
    origins: Dict[str, float] = {}
    dests: Dict[str, float] = {}
    stops: Dict[str, float] = {}
    price_samples: List[int] = []
    time_hours: Dict[int, float] = {}
    vehicle_pref: Dict[str, float] = {}

    def add(m: Dict[str, float], key: Optional[str], w: float):
        k = _norm(key)
        if not k:
            return
        m[k] = m.get(k, 0.0) + w

    def add_vehicle(key: Optional[str], w: float):
        k = _norm(key)
        if not k:
            return
        vehicle_pref[k] = vehicle_pref.get(k, 0.0) + w

    for b in history:
        w = _recency_weight(b.finalized_at, now)

        o = _norm(b.from_stop_name)
        d = _norm(b.to_stop_name)

        if o:
            add(origins, o, 1.2 * w)
            add(stops, o, 1.0 * w)
        if d:
            add(dests, d, 1.2 * w)
            add(stops, d, 1.0 * w)
        if o and d:
            add(route_pairs, f"{o} -> {d}", 1.7 * w)

        tf = _safe_int(b.total_fare, 0)
        if tf > 0:
            price_samples.append(tf)

        dt = _parse_iso_dt(b.finalized_at)
        if dt:
            hr = int(dt.hour)
            time_hours[hr] = time_hours.get(hr, 0.0) + w

        add_vehicle(b.vehicle_company, 1.4 * w)
        add_vehicle(b.vehicle_type, 1.1 * w)
        add_vehicle(b.vehicle_fuel_type, 0.9 * w)
        add_vehicle(b.vehicle_color, 0.5 * w)
        seats = _safe_int(b.vehicle_seats, 0)
        if seats > 0:
            add_vehicle(f"seats:{seats}", 1.0 * w)

    price_med = 0.0
    if price_samples:
        price_samples.sort()
        price_med = float(price_samples[len(price_samples) // 2])

    def topk(m: Dict[str, float], k: int) -> List[str]:
        return [x[0] for x in sorted(m.items(), key=lambda t: t[1], reverse=True)[:k]]

    return {
        "top_route_pairs": topk(route_pairs, 8),
        "top_origins": topk(origins, 8),
        "top_dests": topk(dests, 8),
        "top_stops": topk(stops, 15),
        "price_median": price_med,
        "hour_pref": time_hours,
        "vehicle_pref": vehicle_pref,
    }


def _hour_affinity(dep_iso: Optional[str], hour_pref: Dict[int, float]) -> float:
    if not hour_pref:
        return 0.0
    dep = _parse_iso_dt(dep_iso)
    if not dep:
        return 0.0
    h = int(dep.hour)
    mx = max(hour_pref.values()) if hour_pref else 1.0
    return float(hour_pref.get(h, 0.0) / max(1e-9, mx))


def _price_affinity(price: Optional[int], price_median: float) -> float:
    p = _safe_float(price, 0.0)
    if p <= 0 or price_median <= 0:
        return 0.0

    rel = (p - price_median) / max(1.0, price_median)

    if rel <= 0:
        return math.exp(-0.6 * abs(rel))

    return math.exp(-1.4 * abs(rel))


def _vehicle_affinity(c: CandidateTrip, vehicle_pref: Dict[str, float]) -> float:
    if not vehicle_pref:
        return 0.0
    s = 0.0
    keys = [
        c.vehicle_company,
        c.vehicle_type,
        c.vehicle_fuel_type,
        c.vehicle_color,
    ]
    for k in keys:
        kk = _norm(k)
        if kk:
            s += vehicle_pref.get(kk, 0.0)
    seats = _safe_int(c.vehicle_seats, 0)
    if seats > 0:
        s += vehicle_pref.get(f"seats:{seats}", 0.0)
    return 1.0 - math.exp(-s)


def _stopbreakdown_stop_set(c: CandidateTrip) -> set:
    s = set()
    for b in c.stop_breakdown or []:
        if b.from_stop_name:
            s.add(_norm(b.from_stop_name))
        if b.to_stop_name:
            s.add(_norm(b.to_stop_name))
    return set([x for x in s if x])


def _candidate_segment_set(c: CandidateTrip) -> set:
    segs = set()
    for b in c.stop_breakdown or []:
        fr = _norm(b.from_stop_name)
        to = _norm(b.to_stop_name)
        if fr and to:
            segs.add((fr, to))
    return segs


def _parse_route_pair(rp: str) -> Tuple[str, str]:
    parts = [p.strip() for p in str(rp).split("->")]
    if len(parts) >= 2:
        return _norm(parts[0]), _norm(parts[1])
    return _norm(rp), ""


def _route_affinity(c: CandidateTrip, pref: Dict[str, Any]) -> Tuple[float, List[str]]:
    reasons: List[str] = []
    origin = _norm(c.origin)
    dest = _norm(c.destination)

    pair_score = 0.0
    reverse_pair_score = 0.0
    for rp in pref.get("top_route_pairs", []):
        po, pd = _parse_route_pair(rp)
        if po and pd:
            s_dir = 0.6 * _jaccard(po, origin) + 0.4 * _jaccard(pd, dest)
            s_rev = 0.6 * _jaccard(po, dest) + 0.4 * _jaccard(pd, origin)
            pair_score = max(pair_score, s_dir)
            reverse_pair_score = max(reverse_pair_score, s_rev)

    pair_score = max(pair_score, 0.10 * reverse_pair_score)

    if pair_score > 0.25:
        reasons.append("route_pair_match")

    stop_score = 0.0
    cand_stops = _stopbreakdown_stop_set(c)
    if cand_stops:
        top_stops = pref.get("top_stops", [])
        hit = 0.0
        for ts in top_stops:
            if ts in cand_stops:
                hit += 1.0
        stop_score = min(1.0, hit / 5.0)
        if stop_score > 0.2:
            reasons.append("stop_familiarity")

    seg_score = 0.0
    cand_segs = _candidate_segment_set(c)
    if cand_segs:
        for rp in pref.get("top_route_pairs", []):
            po, pd = _parse_route_pair(rp)
            if po and pd and (po, pd) in cand_segs:
                seg_score = 1.0
                break
        if seg_score > 0.0:
            reasons.append("segment_match")

    od_score = 0.0
    for o in pref.get("top_origins", []):
        od_score = max(od_score, _jaccard(o, origin))
    for d in pref.get("top_dests", []):
        od_score = max(od_score, _jaccard(d, dest))
    if od_score > 0.25:
        reasons.append("origin_dest_match")

    route_total = 0.40 * pair_score + 0.22 * seg_score + 0.23 * stop_score + 0.15 * od_score
    return route_total, reasons


def _price_far_penalty(departure_iso: Optional[str], now: datetime, *, distance_km: float) -> float:
    dep = _parse_iso_dt(departure_iso)
    if not dep:
        return 0.0
    delta_h = (dep - now).total_seconds() / 3600.0
    if delta_h <= 0:
        return 0.0

    p = _distance_time_params(distance_km)
    if delta_h <= p["far_start_h"]:
        return 0.0

    span = max(1e-6, (p["far_end_h"] - p["far_start_h"]))
    return _clamp((delta_h - p["far_start_h"]) / span, 0.0, 1.0)


# Slight weight tweak:
# - increase driver impact
# - reduce price + seats impact
W_ROUTE = 0.46
W_SOON = 0.12
W_DRIVER = 0.18
W_PRICE = 0.07
W_VEH = 0.08
W_HOUR = 0.04
W_SEATS = 0.01
W_FAR_PEN = 0.06


def _score_candidate(user: UserPayload, pref: Dict[str, Any], now: datetime, c: CandidateTrip) -> RankedItem:
    reasons: List[str] = []

    gpen = _gender_penalty(user.gender, c.gender_preference)
    if gpen > 0:
        reasons.append("gender_mismatch")

    route_score, route_reasons = _route_affinity(c, pref)
    reasons.extend(route_reasons)

    dist_km = _trip_distance_km(c)

    soon = _soonness_score(c.departure_time, now, distance_km=dist_km)
    if soon > 0.85:
        reasons.append("soon_departure")

    far_pen = _price_far_penalty(c.departure_time, now, distance_km=dist_km)
    if far_pen > 0.35:
        reasons.append("far_departure")

    dr = max(0.0, min(5.0, _safe_float(c.driver_rating, 0.0)))
    driver_score = dr / 5.0
    if driver_score > 0.7:
        reasons.append("high_driver_rating")

    price_score = _price_affinity(c.price_per_seat, float(pref.get("price_median") or 0.0))
    if price_score > 0.5:
        reasons.append("price_fit")

    neg_bonus = 0.0
    if c.is_negotiable is True:
        neg_bonus = 0.15
        reasons.append("negotiable")

    veh = _vehicle_affinity(c, pref.get("vehicle_pref") or {})
    if veh > 0.3:
        reasons.append("vehicle_match")

    hour_aff = _hour_affinity(c.departure_time, pref.get("hour_pref") or {})
    if hour_aff > 0.4:
        reasons.append("time_of_day_match")

    seats = _safe_int(c.available_seats, 0)
    seats_score = 1.0 - math.exp(-seats / 3.0) if seats > 0 else 0.0

    score = (
        W_ROUTE * route_score +
        W_SOON * soon +
        W_DRIVER * driver_score +
        W_PRICE * price_score +
        W_VEH * veh +
        W_HOUR * hour_aff +
        W_SEATS * seats_score +
        neg_bonus
        - W_FAR_PEN * far_pen
    )

    score = max(0.0, min(1.0, score * (1.0 - gpen)))
    return RankedItem(trip_id=c.trip_id, score=float(score), reasons=reasons)


class _MLRankerNamespace:
    UserPayload = UserPayload
    HistoryBooking = HistoryBooking
    CandidateTrip = CandidateTrip
    RankedItem = RankedItem
    _parse_iso_dt = staticmethod(_parse_iso_dt)
    datetime = datetime
    _build_user_pref = staticmethod(_build_user_pref)
    _score_candidate = staticmethod(_score_candidate)


mlranker = _MLRankerNamespace()

# -----------------------------
# Helpers
# -----------------------------
def parse_json_safe(s):
    if s is None:
        return None
    s = str(s)
    if s.strip() == "":
        return None
    return json.loads(s)


def dcg(rels):
    score = 0.0
    for i, r in enumerate(rels, start=1):
        score += (2**r - 1) / math.log2(i + 1)
    return score


def ndcg_at_k(labels, k):
    rels = labels[:k]
    ideal = sorted(labels, reverse=True)[:k]
    denom = dcg(ideal)
    return (dcg(rels) / denom) if denom > 0 else 0.0


def mrr_at_k(labels, k):
    for i, lbl in enumerate(labels[:k], start=1):
        if lbl == 1:
            return 1.0 / i
    return 0.0


def hit_at_k(labels, k):
    return 1.0 if any(lbl == 1 for lbl in labels[:k]) else 0.0


def map_at_k(labels, k):
    hits = 0
    sum_prec = 0.0
    for i, lbl in enumerate(labels[:k], start=1):
        if lbl == 1:
            hits += 1
            sum_prec += hits / i
    return (sum_prec / hits) if hits > 0 else 0.0


def _thin_candidate_view(c: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "origin": c.get("origin"),
        "destination": c.get("destination"),
        "departure_time": c.get("departure_time"),
        "price_per_seat": c.get("price_per_seat"),
        "available_seats": c.get("available_seats"),
        "driver_rating": c.get("driver_rating"),
        "gender_preference": c.get("gender_preference"),
        "is_negotiable": c.get("is_negotiable"),
        "vehicle_company": c.get("vehicle_company"),
        "vehicle_type": c.get("vehicle_type"),
        "vehicle_fuel_type": c.get("vehicle_fuel_type"),
        "vehicle_color": c.get("vehicle_color"),
        "vehicle_seats": c.get("vehicle_seats"),
        "total_distance_km": c.get("total_distance_km"),
        "total_duration_minutes": c.get("total_duration_minutes"),
    }


# -----------------------------
# Load CSV
# -----------------------------
usecols = [
    "event_id",
    "user_id",
    "user_gender",
    "passenger_rating",
    "now_iso",
    "history_bookings_json",
    "candidate_json",
    "label",
]

df = pd.read_csv(CSV_PATH, usecols=usecols)

if MAX_EVENTS is not None:
    keep_events = df["event_id"].drop_duplicates().head(MAX_EVENTS)
    df = df[df["event_id"].isin(set(keep_events))]

# -----------------------------
# Evaluate
# -----------------------------
metrics = []
worst_events_debug = []

for eid, ev in df.groupby("event_id", sort=False):
    if MAX_CANDIDATES_PER_EVENT is not None and len(ev) > MAX_CANDIDATES_PER_EVENT:
        pos = ev[ev["label"] == 1]
        neg = ev[ev["label"] != 1].head(max(0, MAX_CANDIDATES_PER_EVENT - len(pos)))
        ev = pd.concat([pos, neg], axis=0)

    r0 = ev.iloc[0]
    user = mlranker.UserPayload(
        user_id=int(r0["user_id"]),
        gender=str(r0["user_gender"]),
        passenger_rating=float(r0.get("passenger_rating", 0.0) or 0.0),
    )

    now = mlranker._parse_iso_dt(str(r0["now_iso"]))
    if now is None:
        now = datetime.now(timezone.utc)

    hist_list = parse_json_safe(r0["history_bookings_json"]) or []
    history_objs = [mlranker.HistoryBooking(**b) for b in hist_list]
    pref = mlranker._build_user_pref(history_objs, now)

    cand_by_trip: Dict[str, Dict[str, Any]] = {}
    scored = []

    for _, row in ev.iterrows():
        cand = parse_json_safe(row["candidate_json"]) or {}
        cobj = mlranker.CandidateTrip(**cand)
        item = mlranker._score_candidate(user, pref, now, cobj)

        tid = str(cobj.trip_id)
        lbl = int(row["label"])
        cand_by_trip[tid] = cand
        scored.append((tid, float(item.score), lbl, list(item.reasons)))

    scored.sort(key=lambda x: x[1], reverse=True)
    labels_sorted = [lbl for _, _, lbl, _ in scored]

    booked_rank = next((i for i, lbl in enumerate(labels_sorted, start=1) if lbl == 1), 10**9)

    metrics.append({
        "event_id": eid,
        "candidates": len(scored),
        "booked_rank": booked_rank,
        "MRR@10": mrr_at_k(labels_sorted, 10),
        "NDCG@10": ndcg_at_k(labels_sorted, 10),
        "MAP@10": map_at_k(labels_sorted, 10),
        "Hit@5": hit_at_k(labels_sorted, 5),
        "Hit@10": hit_at_k(labels_sorted, 10),
    })

    if booked_rank >= WORST_EVENT_MIN_BOOKED_RANK:
        pos_trip_id = next((tid for tid, _s, lbl, _r in scored if lbl == 1), None)
        worst_events_debug.append({
            "event_id": eid,
            "booked_rank": booked_rank,
            "user": {
                "user_id": int(r0["user_id"]),
                "gender": str(r0["user_gender"]),
                "passenger_rating": float(r0.get("passenger_rating", 0.0) or 0.0),
            },
            "now_iso": str(r0["now_iso"]),
            "pos_trip_id": pos_trip_id,
            "pos_candidate": cand_by_trip.get(str(pos_trip_id)) if pos_trip_id else None,
            "topk": [
                {
                    "trip_id": tid,
                    "label": lbl,
                    "score": s,
                    "reasons": reasons,
                    "candidate": cand_by_trip.get(tid),
                }
                for tid, s, lbl, reasons in scored[:PRINT_TOP_K]
            ],
        })

res = pd.DataFrame(metrics)

print("Events:", len(res))
print("\nAverages:")
print(res[["Hit@5", "Hit@10", "MRR@10", "NDCG@10", "MAP@10"]].mean())
print("\nBooked rank:")
print("Median booked_rank:", res["booked_rank"].median())
print("Mean booked_rank:", res["booked_rank"].mean())

print("\nWorst 10 events by booked_rank:")
print(res.sort_values("booked_rank", ascending=False).head(10)[["event_id", "candidates", "booked_rank"]])

# -----------------------------
# Detailed debug: worst events
# -----------------------------
worst_events_debug.sort(key=lambda x: x["booked_rank"], reverse=True)
print("\nDetailed debug for worst events (up to", PRINT_WORST_N_EVENTS, "):")

for d in worst_events_debug[:PRINT_WORST_N_EVENTS]:
    print("\n---")
    print("event_id:", d["event_id"], "booked_rank:", d["booked_rank"])
    print("user:", d["user"])
    print("now_iso:", d["now_iso"])
    print("pos_trip_id:", d["pos_trip_id"])
    print("pos_candidate:", _thin_candidate_view(d["pos_candidate"] or {}))

    print("\nTop", PRINT_TOP_K, "candidates:")
    for t in d["topk"]:
        c = t["candidate"] or {}
        print(
            "  - trip_id:", t["trip_id"],
            "label=", t["label"],
            "score=", round(float(t["score"]), 6),
        )
        print("    reasons:", t["reasons"])
        print("    candidate:", _thin_candidate_view(c))


In [ ]:
# Sanity-check debug for the worst event

# Uses the exact same weights as the ranker cell.

def _debug_components(user, pref, now, cand_dict):
    cobj = mlranker.CandidateTrip(**(cand_dict or {}))
    dist_km = _trip_distance_km(cobj)

    route_score, _route_reasons = _route_affinity(cobj, pref)
    soon = _soonness_score(cobj.departure_time, now, distance_km=dist_km)
    far_pen = _price_far_penalty(cobj.departure_time, now, distance_km=dist_km)
    dr = max(0.0, min(5.0, _safe_float(cobj.driver_rating, 0.0)))
    driver_score = dr / 5.0
    price_score = _price_affinity(cobj.price_per_seat, float(pref.get("price_median") or 0.0))
    veh = _vehicle_affinity(cobj, pref.get("vehicle_pref") or {})
    hour_aff = _hour_affinity(cobj.departure_time, pref.get("hour_pref") or {})
    seats = _safe_int(cobj.available_seats, 0)
    seats_score = 1.0 - math.exp(-seats / 3.0) if seats > 0 else 0.0
    gpen = _gender_penalty(user.gender, cobj.gender_preference)

    neg_bonus = 0.15 if cobj.is_negotiable is True else 0.0

    score_raw = (
        W_ROUTE * route_score +
        W_SOON * soon +
        W_DRIVER * driver_score +
        W_PRICE * price_score +
        W_VEH * veh +
        W_HOUR * hour_aff +
        W_SEATS * seats_score +
        neg_bonus
        - W_FAR_PEN * far_pen
    )
    score_final = max(0.0, min(1.0, score_raw * (1.0 - gpen)))

    return {
        "distance_km": dist_km,
        "soon": soon,
        "far_pen": far_pen,
        "route_score": route_score,
        "driver_score": driver_score,
        "price_score": price_score,
        "vehicle_aff": veh,
        "hour_aff": hour_aff,
        "seats_score": seats_score,
        "neg_bonus": neg_bonus,
        "gender_pen": gpen,
        "score_raw": score_raw,
        "score_final": score_final,
    }


def print_worst_event_debug(target_event_id=None):
    if res is None or res.empty:
        print("No evaluation results found. Run the evaluation cell first.")
        return

    worst_row = res.sort_values("booked_rank", ascending=False).iloc[0]
    worst_eid = str(worst_row["event_id"])

    eid = str(target_event_id) if target_event_id else worst_eid

    ev = df[df["event_id"] == eid]
    if ev.empty:
        print("Event not found:", eid)
        print("Worst event in current results:", worst_eid)
        return

    r0 = ev.iloc[0]
    user = mlranker.UserPayload(
        user_id=int(r0["user_id"]),
        gender=str(r0["user_gender"]),
        passenger_rating=float(r0.get("passenger_rating", 0.0) or 0.0),
    )

    now = mlranker._parse_iso_dt(str(r0["now_iso"]))
    if now is None:
        now = datetime.now(timezone.utc)

    hist_list = parse_json_safe(r0["history_bookings_json"]) or []
    history_objs = [mlranker.HistoryBooking(**b) for b in hist_list]
    pref = mlranker._build_user_pref(history_objs, now)

    scored = []
    cand_by_trip = {}
    for _, row in ev.iterrows():
        cand = parse_json_safe(row["candidate_json"]) or {}
        cobj = mlranker.CandidateTrip(**cand)
        item = mlranker._score_candidate(user, pref, now, cobj)
        tid = str(cobj.trip_id)
        lbl = int(row["label"])
        cand_by_trip[tid] = cand
        scored.append((tid, float(item.score), lbl, list(item.reasons)))

    scored.sort(key=lambda x: x[1], reverse=True)
    labels_sorted = [lbl for _, _, lbl, _ in scored]
    booked_rank = next((i for i, lbl in enumerate(labels_sorted, start=1) if lbl == 1), 10**9)

    pos_trip_id = next((tid for tid, _s, lbl, _r in scored if lbl == 1), None)

    print("\n=== WORST EVENT DEBUG ===")
    print("event_id:", eid, "booked_rank:", booked_rank, "candidates:", len(scored))
    print("user:", {"user_id": user.user_id, "gender": user.gender, "passenger_rating": user.passenger_rating})
    print("now_iso:", str(r0["now_iso"]))

    if pos_trip_id:
        pos_cand = cand_by_trip.get(str(pos_trip_id))
        pos_comp = _debug_components(user, pref, now, pos_cand)
        print("\nBOOKED (label=1):", pos_trip_id)
        print("  candidate:", _thin_candidate_view(pos_cand or {}))
        print("  components:", {k: (round(v, 6) if isinstance(v, float) else v) for k, v in pos_comp.items()})

    print("\nTOP", PRINT_TOP_K, "CANDIDATES:")
    for rank_idx, (tid, s, lbl, reasons) in enumerate(scored[:PRINT_TOP_K], start=1):
        cand = cand_by_trip.get(tid)
        comp = _debug_components(user, pref, now, cand)
        print("\n#", rank_idx, "trip_id:", tid, "label=", lbl, "score=", round(float(s), 6))
        print("  reasons:", reasons)
        print("  candidate:", _thin_candidate_view(cand or {}))
        print("  components:", {k: (round(v, 6) if isinstance(v, float) else v) for k, v in comp.items()})

    if pos_trip_id:
        top_neg = next(((tid, s, reasons) for tid, s, lbl, reasons in scored if lbl == 0), None)
        if top_neg:
            neg_tid, neg_s, neg_reasons = top_neg
            pos_s = next((s for tid, s, lbl, _r in scored if lbl == 1 and tid == pos_trip_id), None)
            print("\n=== QUICK COMPARISON ===")
            print("pos_trip_id:", pos_trip_id, "pos_score:", round(float(pos_s or 0.0), 6))
            print("top_negative_trip_id:", neg_tid, "neg_score:", round(float(neg_s), 6))
            print("top_negative_reasons:", neg_reasons)


print_worst_event_debug("E1916")
